In [6]:
# 1. Setup and imports
!pip install -q sentence-transformers gensim datasets

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
import gensim.downloader as api
from datasets import load_dataset

# Reproducibility seed
np.random.seed(42)

In [7]:
# 2. Load official STS-Benchmark test set via Hugging Face
dataset = load_dataset("mteb/stsbenchmark-sts", split="test")

# Convert to DataFrame and standardize column names
test_df = pd.DataFrame({
    'score': dataset['score'],
    'sent1': dataset['sentence1'],
    'sent2': dataset['sentence2']
})

print(f"Loaded {len(test_df)} test sentence pairs.")
test_df.head(3)

README.md:   0%|          | 0.00/5.67k [00:00<?, ?B/s]

train.jsonl.gz: reconstructing file:   0%|          |  0.00B /  278kB            

train.jsonl.gz: downloading bytes:           |  0.00B            

validation.jsonl.gz: reconstructing file:   0%|          |  0.00B / 86.4kB            

validation.jsonl.gz: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Loaded 1379 test sentence pairs.


,score,sent1,sent2
0,2.5,A girl is styling her hair.,A girl is brushing her hair.
1,3.6,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.
2,5.0,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.


In [8]:
# 3. GloVe 300d Baseline Model
print("Loading GloVe 300d embeddings...")
glove = api.load("glove-wiki-gigaword-300")

def get_sentence_vector(sentence):
    tokens = [w.lower() for w in sentence.split() if w.lower() in glove]
    if not tokens:
        return np.zeros(300)
    return np.mean([glove[w] for w in tokens], axis=0)

# Compute sentence vectors
vecs1 = np.array([get_sentence_vector(s) for s in test_df['sent1']])
vecs2 = np.array([get_sentence_vector(s) for s in test_df['sent2']])

# Vectorized Cosine Similarity
norms1 = np.linalg.norm(vecs1, axis=1)
norms2 = np.linalg.norm(vecs2, axis=1)
valid = (norms1 > 0) & (norms2 > 0)

glove_sim = np.zeros(len(test_df))
glove_sim[valid] = np.sum(vecs1[valid] * vecs2[valid], axis=1) / (norms1[valid] * norms2[valid])

# Scale cosine [-1, 1] to STS score range [0, 5]
test_df['glove_pred'] = np.clip((glove_sim + 1) / 2 * 5, 0, 5)

glove_corr, _ = spearmanr(test_df['score'], test_df['glove_pred'])
print(f"GloVe 300d Spearman correlation: {glove_corr:.4f}")

Loading GloVe 300d embeddings...
[==================================================] 100.0% 376.1/376.1MB downloaded
GloVe 300d Spearman correlation: 0.3184


In [9]:
# 4. Sentence-BERT (all-MiniLM-L6-v2) Evaluation
print("Loading Sentence-BERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode sentences to normalized dense vectors
emb1 = sbert_model.encode(test_df['sent1'].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb2 = sbert_model.encode(test_df['sent2'].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# Cosine similarity for normalized vectors is just the dot product
sbert_sim = np.sum(emb1 * emb2, axis=1)

# Scale cosine [-1, 1] to STS score range [0, 5]
test_df['sbert_pred'] = np.clip((sbert_sim + 1) / 2 * 5, 0, 5)

sbert_corr, _ = spearmanr(test_df['score'], test_df['sbert_pred'])
print(f"SBERT Spearman correlation: {sbert_corr:.4f}")

Loading Sentence-BERT model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

SBERT Spearman correlation: 0.8203


In [10]:
# 5. Statistical Significance Testing (Paired Bootstrap)
def paired_bootstrap_test(y_true, y_pred_glove, y_pred_sbert, n_bootstraps=1000):
    n = len(y_true)
    diffs = []

    for _ in range(n_bootstraps):
        # Sample with replacement
        indices = np.random.choice(n, size=n, replace=True)
        corr_glove, _ = spearmanr(y_true[indices], y_pred_glove[indices])
        corr_sbert, _ = spearmanr(y_true[indices], y_pred_sbert[indices])
        diffs.append(corr_sbert - corr_glove)

    diffs = np.array(diffs)
    p_value = np.mean(diffs <= 0)
    ci_lower = np.percentile(diffs, 2.5)
    ci_upper = np.percentile(diffs, 97.5)

    return p_value, (ci_lower, ci_upper)

p_val, (ci_low, ci_high) = paired_bootstrap_test(
    test_df['score'].values,
    test_df['glove_pred'].values,
    test_df['sbert_pred'].values
)

print(f"Empirical Delta (SBERT - GloVe): {sbert_corr - glove_corr:.4f}")
print(f"95% Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"p-value: {p_val:.5f}")

Empirical Delta (SBERT - GloVe): 0.5019
95% Bootstrap CI: [0.4489, 0.5488]
p-value: 0.00000


In [11]:
# 6. Qualitative Error Mining (Top Divergent Cases)
test_df['glove_err'] = np.abs(test_df['score'] - test_df['glove_pred'])
test_df['sbert_err'] = np.abs(test_df['score'] - test_df['sbert_pred'])

# Top 15 cases where GloVe fails significantly worse than SBERT
glove_fails = test_df.sort_values(
    by=['glove_err'], ascending=False
).head(15).copy()
glove_fails['failure_type'] = 'GloVe Failure'

# Top 15 cases with highest SBERT error
sbert_fails = test_df.sort_values(
    by=['sbert_err'], ascending=False
).head(15).copy()
sbert_fails['failure_type'] = 'SBERT Failure'

# Combine and export raw mined cases
mined_cases = pd.concat([glove_fails, sbert_fails], ignore_index=True)
columns_to_keep = ['failure_type', 'sent1', 'sent2', 'score', 'glove_pred', 'sbert_pred', 'glove_err', 'sbert_err']
mined_cases = mined_cases[columns_to_keep]

# Save to CSV
mined_cases.to_csv("sts_error_mining_raw.csv", index=False)
print(f"Successfully exported {len(mined_cases)} candidate failure cases to 'sts_error_mining_raw.csv'.")

Successfully exported 30 candidate failure cases to 'sts_error_mining_raw.csv'.
